
# 🐍 Programación II – Semana 2  
### **Encapsulamiento, Atributos Privados y Métodos de Acceso (@property)**
**Profesor:** Mg. José Andrés Zúñiga Cazorla  
**Carrera:** Ciencia de Datos e IA – UNACH  
**Fecha:** 2025-10-13

> Objetivo: Comprender y aplicar el **encapsulamiento** en Python usando **atributos privados** y **propiedades** para proteger el estado de los objetos.



## 1. ¿Qué es el encapsulamiento?
El encapsulamiento **protege** los datos de una clase y **controla** cómo se accede/modifica su estado.
En Python no hay privacidad estricta; se usan **convenciones**:
- `_atributo`  (protegido / para uso interno)
- `__atributo` (privado con *name mangling* → `_Clase__atributo`)


### 1.1. Name mangling en acción

In [4]:

class Demo:
    def __init__(self):
        self.publico = "ok"
        self._interno = "usar con cuidado"
        self.__privado = "no tocar directamente"

    def revelar(self):
        # Acceso interno permitido
        return self.__privado

d = Demo()
print("publico:", d.publico)
print("_interno:", d._interno)
print("revelar():", d.revelar())

# ¿Qué atributos existen realmente?
print([a for a in dir(d) if "priv" in a])
# Intentar acceder directamente descomenta y ejecuta:
# d.__privado


publico: ok
_interno: usar con cuidado
revelar(): no tocar directamente
['_Demo__privado']



> **Nota:** `__privado` se renombra internamente a `_Demo__privado` para evitar colisiones en subclases.


## 2. Getters y Setters idiomáticos con `@property`


En Python preferimos **atributos públicos simples** y usamos `@property` **solo** cuando hay necesidad de:
- validación de datos,
- cálculo perezoso (*lazy*),
- *logging* o *caching*,
- mantener compatibilidad del API.


In [5]:

class Persona:
    def __init__(self, nombre: str, edad: int):
        self._nombre = nombre
        self._edad = edad

    @property
    def nombre(self) -> str:
        "Getter elegante para nombre"
        return self._nombre

    @nombre.setter
    def nombre(self, valor: str):
        "Validación simple"
        if not valor or not valor.strip():
            raise ValueError("El nombre no puede estar vacío")
        self._nombre = valor.strip()

    @property
    def mayoria_edad(self) -> bool:
        "Propiedad de solo lectura derivada"
        return self._edad >= 18

p = Persona("Ana", 20)
print(p.nombre, "→ mayor de edad:", p.mayoria_edad)
p.nombre = "  Ana María  "
print("Normalizado:", p.nombre)


Ana → mayor de edad: True
Normalizado: Ana María


### 2.1. Propiedades con *caching* (ejemplo práctico)

In [6]:

class DataSetCostoso:
    def __init__(self, n: int = 2_000_000):
        self.n = n
        self._suma_cache = None  # aún no calculado

    @property
    def suma(self):
        # Calcula solo la primera vez (lazy + cache)
        if self._suma_cache is None:
            total = 0
            for i in range(self.n):
                total += i
            self._suma_cache = total
        return self._suma_cache

ds = DataSetCostoso(100_000)
print("Primera vez (tarda algo) →", ds.suma)
print("Segunda vez (usando cache) →", ds.suma)


Primera vez (tarda algo) → 4999950000
Segunda vez (usando cache) → 4999950000


## 3. Buenas prácticas (checklist)


- ✅ Usa `@property` *solo si* necesitas lógica al acceder/modificar.  
- ✅ Documenta con **docstrings**.  
- ✅ Usa `_atributo` (protegido) y `__atributo` (privado) con intención.  
- ❌ Evita getters/setters “vacíos” (no idiomáticos en Python).  
- ✅ Expón un **API estable**; cambia la implementación sin romper al usuario.


## 4. Mini–reto guiado


**Instrucciones:** Implementa una clase `CuentaBancaria` que:  
1) Guarde `__saldo` como atributo privado.  
2) Tenga métodos `depositar(monto)` y `retirar(monto)` con validaciones.  
3) Exponga una propiedad `saldo` **solo lectura**.  
4) Agrega *docstrings* y pruebas rápidas con `assert`.


In [2]:
class CuentaBancaria:
    def __init__(self, saldo=0):
        self.__saldo = saldo
    """Depositar una cantidad positiva
    monto: la cantidad a depositar,
    raise ValueError: si el monto en menor a cero."""
    def depositar(self, monto):
        if monto <= 0:
            raise ValueError("Error, el monto a depositar debe ser positivo")
        self.__saldo += monto

    """Retira la cantidad si hay monto suficiente y si es positivo"""
    def retirar(self, monto):
        if monto <= 0:
            raise ValueError("Error, el monto a retirar debe ser positivo")
        if monto > self.__saldo:
            raise ValueError("Saldo insuficiente")
        self.__saldo -= monto
    """Solo de lectura para mostrar el saldo actual"""
    @property
    def saldo(self):
        return self.__saldo

In [5]:
cuenta = CuentaBancaria(2000)
# Depositar, suma al saldo
cuenta.depositar(100)
assert cuenta.saldo == 2100
# Retirar, resta al saldo actual : 2100-700=1400
cuenta.retirar(700)
assert cuenta.saldo == 1400

try:
    cuenta.retirar(1500)
    assert False, "Saldo insuficiente, error"
except ValueError:
    pass

try:
    cuenta.depositar(-180)
    assert False, "Saldo insuficiente, error"
except ValueError:
    pass

try:
    cuenta.saldo = 5000
    assert False, "Error porque es solo de lectura"
except AttributeError:
    pass

